# 第 3 章 §3.3 · Parallelization（Google ADK 版）

书里这份是**代码片段**（无 import、无 Runner），下面补了 import，所以第一格可以离线执行——
**构造 agent 不花 token**。第二格才真正 `runner.run`，那一步要额度。

形状和 LangChain 版一样是**扇出 → 汇合**，但并行的单位是一个**完整 agent**（多轮推理 + 工具），
数据不是靠函数返回值传，而是靠 **session state**：研究员 `output_key=` 写，汇总用 `{占位符}` 读。

kernel 必须选 **`Python (agentic-adk)`**。两个实测发现（ParallelAgent 已弃用、免费层 5 RPM 跑不动）
和 LangChain 版的对照都在 `../../ch03_parallelization/README.md`。

In [1]:
# 【适配】原书是片段，没有 import（原文交代要放进 agent.py）；这里补上，便于离线构造。
# ⚠️ ParallelAgent / SequentialAgent 在 ADK 2.13 已标记弃用（官方让换 Workflow），仍能用。
import os

from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.tools import google_search

# 【适配】书里 gemini-2.0-flash 已下线
GEMINI_MODEL = os.getenv("GOOGLE_MODEL", "gemini-3.1-flash-lite")


# ★ 关键机制一：output_key
#   output_key="renewable_energy_result" 的意思是：这个 agent 的最终输出会被写进
#   【session state】的同名键。这是 agent 之间传递数据的唯一通道 —— 不是函数返回值。
#
# ★ 关键机制二：tools=[google_search]
#   给了 agent 联网搜索能力，也让它变成"多轮"：先决定搜什么 → 搜 → 再总结。
#   所以一次并行不是 3 次模型调用，而可能是 6~9 次（每个 agent 至少 2 轮）。
#   免费层 5 RPM 的情况下，这一节基本跑不动。

researcher_agent_1 = LlmAgent(
    name="RenewableEnergyResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in energy.
Research the latest advancements in 'renewable energy sources'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[google_search],
    output_key="renewable_energy_result"      # Store result in state for the merger agent
)

researcher_agent_2 = LlmAgent(
    name="EVResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in transportation.
Research the latest developments in 'electric vehicle technology'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[google_search],
    output_key="ev_technology_result"
)

researcher_agent_3 = LlmAgent(
    name="CarbonCaptureResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Research the current state of 'carbon capture methods'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[google_search],
    output_key="carbon_capture_result"
)

# ★ ParallelAgent 本身不做任何计算，只做调度：把 sub_agents 同时启动，
#   等【全部完成】后才算自己完成（fan-in 语义）。
#   和 RunnableParallel 的区别在粒度：LangChain 并行的是"链"，ADK 并行的是"完整 agent"。
parallel_research_agent = ParallelAgent(
    name="ParallelWebResearchAgent",
    sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
    description="Runs multiple research agents in parallel to gather information."
)

# ★ 关键机制三：instruction 里的 {renewable_energy_result} 这类占位符，会被 ADK 用
#   session state 里同名键的值替换掉。两者靠【字符串名字】约定 —— 和 LangChain 的
#   字典键名是同一类契约，代价也一样：名字写错不会编译报错，只会在运行时填不进去。
#
#   那句 "MUST be grounded *exclusively* on the information provided" 是在抑制模型
#   自行补充外部知识（幻觉），属于第 18 章 guardrails 的手段。
merger_agent = LlmAgent(
    name="SynthesisAgent",
    model=GEMINI_MODEL,
    instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

 Your primary task is to synthesize the following research summaries, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

 **Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the 'Input Summaries' below. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

 **Input Summaries:**

 *   **Renewable Energy:**
     {renewable_energy_result}

 *   **Electric Vehicles:**
     {ev_technology_result}

 *   **Carbon Capture:**
     {carbon_capture_result}

 **Output Format:**

 ## Summary of Recent Sustainable Technology Advancements

 ### Renewable Energy Findings
 (Based on RenewableEnergyResearcher's findings)
 [Synthesize and elaborate *only* on the renewable energy input summary provided above.]

 ### Electric Vehicle Findings
 (Based on EVResearcher's findings)
 [Synthesize and elaborate *only* on the EV input summary provided above.]

 ### Carbon Capture Findings
 (Based on CarbonCaptureResearcher's findings)
 [Synthesize and elaborate *only* on the carbon capture input summary provided above.]

 ### Overall Conclusion
 [Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

 Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
 """,
    description="Combines research findings from parallel agents into a structured, cited report, strictly grounded on provided inputs.",
)

# 用两个容器 agent 把"并行"和"串行"拼成一条流水线：
#   SequentialAgent
#     ├── ParallelAgent (三个研究员并发)
#     └── LlmAgent 汇总   ← 通过 {占位符} 从 state 读
# root_agent 这个名字是 ADK 的约定：adk web / adk run 会去找它作为入口。
sequential_pipeline_agent = SequentialAgent(
    name="ResearchAndSynthesisPipeline",
    sub_agents=[parallel_research_agent, merger_agent],
    description="Coordinates parallel research and synthesizes the results."
)

root_agent = sequential_pipeline_agent
print("✅ 结构构造完毕（没有调用任何模型）：", root_agent.name)
print("   子 agent:", [getattr(a, "name", "?") for a in root_agent.sub_agents])

✅ 结构构造完毕（没有调用任何模型）： ResearchAndSynthesisPipeline
   子 agent: ['ParallelWebResearchAgent', 'SynthesisAgent']


/tmp/ipykernel_2971317/4079460953.py:66: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_agent = ParallelAgent(
/tmp/ipykernel_2971317/4079460953.py:127: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  sequential_pipeline_agent = SequentialAgent(


In [2]:
# 真正跑起来。⚠️ 这一步要额度：3 个研究员各自多轮 + 1 次汇总 ≈ 6~9 次模型调用，
# 免费层 5 RPM 基本一定 429。下面只跑一次，失败会打印原因而不是崩掉。
import asyncio
import uuid

from google.adk.runners import InMemoryRunner
from google.genai import types

SUBJECT = "Summarize recent sustainable technology advancements."


def run_pipeline(pipeline, subject: str):
    runner = InMemoryRunner(pipeline)
    session_id = str(uuid.uuid4())
    # 【适配】create_session 在 ADK 2.13 是协程，同步上下文用 create_session_sync
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="user_1", session_id=session_id
    )

    finals = []
    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=subject)]),
    ):
        if event.is_final_response() and event.content:
            text = (
                event.content.text
                if getattr(event.content, "text", None)
                else "".join(p.text for p in (event.content.parts or []) if getattr(p, "text", None))
            )
            finals.append((getattr(event, "author", "?"), text))
    return finals


try:
    for author, text in run_pipeline(sequential_pipeline_agent, SUBJECT):
        print(f"\n--- [{author}] ---")
        print(text[:800])
except Exception as e:
    msg = str(e)
    # 取一行有信息量的：Google 的 429 首行往往只是 "On how to mitigate this issue..."
    lines = [ln.strip() for ln in msg.splitlines() if ln.strip()]
    head = next(
        (ln for ln in lines if any(k in ln for k in ("429", "RESOURCE_EXHAUSTED", "quota", "503", "UNAVAILABLE"))),
        lines[0] if lines else type(e).__name__,
    )[:300]
    print(f"❌ {type(e).__name__}: {head}")
    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
        print("   → 免费层 5 RPM 撞限流，这一节要跑通需要付费额度或更强的模型桶。")

Deprecated. Please migrate to the async method.


❌ _ResourceExhaustedError: https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted
   → 免费层 5 RPM 撞限流，这一节要跑通需要付费额度或更强的模型桶。
